# 001 — Fine-Tuning with Unsloth & QLoRA

Fine-tune a pre-trained LLM using QLoRA (4-bit quantized LoRA) with the
Unsloth library for accelerated training. Uses Gemma 3 4B IT on the Alpaca
instruction dataset by default.

**Lifecycle stage:** seedling (model-garden)

All code is self-contained in this notebook — no external library imports
from a shared `src/` package.

In [ ]:
# ---------------------------------------------------------------------------
# Papermill parameters  (this cell is tagged "parameters")
# ---------------------------------------------------------------------------

# Model
model_name: str = "unsloth/gemma-3-4b-it"  # Hugging Face model ID
dataset_name: str = "yahma/alpaca-cleaned"  # Hugging Face dataset ID
max_seq_length: int = 2048
load_in_4bit: bool = True

# LoRA configuration
lora_r: int = 16           # LoRA rank
lora_alpha: int = 32       # LoRA alpha scaling factor
lora_dropout: float = 0.05
target_modules: list[str] = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Training
max_steps: int = 60                        # quick demo; increase for real runs
per_device_train_batch_size: int = 4
gradient_accumulation_steps: int = 4
learning_rate: float = 2e-4
warmup_steps: int = 10
logging_steps: int = 5
lr_scheduler_type: str = "cosine"

# Dataset
dataset_sample_size: int = 1000
test_size: float = 0.1
seed: int = 42

# Inference test prompts
test_prompts: list[str] = [
    "Explain the difference between a list and a tuple in Python.",
    "Write a haiku about machine learning.",
    "What are three benefits of exercise?",
]

# Model saving
save_merged_model: bool = True

# Output paths
metrics_json_path: str = "outputs/metrics/metrics.json"
model_output_path: str = "outputs/models/lora_adapter"
merged_output_path: str = "outputs/models/merged_model"
plots_dir: str = "outputs/plots"
executed_notebook_path: str | None = None

In [ ]:
# ---------------------------------------------------------------------------
# Imports & setup
# ---------------------------------------------------------------------------
import json
import math
import os
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

# Ensure output dirs exist
for d in ["outputs/runs", "outputs/plots", "outputs/models", "outputs/metrics"]:
    Path(d).mkdir(parents=True, exist_ok=True)

run_start = datetime.now(timezone.utc)
print(f"Run started at {run_start.isoformat()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1 — Load Base Model

Load the pre-trained model with 4-bit quantization (QLoRA). This reduces
memory usage by ~4x compared to full precision, making it possible to
fine-tune large models on consumer GPUs.

In [ ]:
# ---------------------------------------------------------------------------
# Load pre-trained model + tokenizer
# ---------------------------------------------------------------------------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Max sequence length: {max_seq_length}")
print(f"Quantization: {'4-bit' if load_in_4bit else 'none'}")

## 2 — Configure LoRA Adapters

LoRA (Low-Rank Adaptation) adds small trainable matrices to the model's
attention and feed-forward layers. Only these adapters are trained, keeping
the base model frozen.

Key parameters:
- **rank (r):** Dimension of the low-rank matrices. Higher = more capacity but more memory.
- **alpha:** Scaling factor. The effective learning rate scales as `alpha / r`.
- **target_modules:** Which layers to attach adapters to.

In [ ]:
# ---------------------------------------------------------------------------
# Attach LoRA adapters
# ---------------------------------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=target_modules,
    use_gradient_checkpointing="unsloth",
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
pct = 100.0 * trainable_params / total_params

print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({pct:.2f}%)")
print(f"LoRA rank: {lora_r}, alpha: {lora_alpha}, dropout: {lora_dropout}")
print(f"Target modules: {target_modules}")

## 3 — Load and Prepare Dataset

Load the Alpaca instruction-following dataset, subsample for quick
experimentation, and format each example with the Alpaca prompt template.

In [ ]:
# ---------------------------------------------------------------------------
# Load and subsample dataset
# ---------------------------------------------------------------------------
raw_dataset = load_dataset(dataset_name, split="train")
print(f"Full dataset size: {len(raw_dataset):,}")

if dataset_sample_size and dataset_sample_size < len(raw_dataset):
    raw_dataset = raw_dataset.shuffle(seed=seed).select(range(dataset_sample_size))
    print(f"Subsampled to: {len(raw_dataset):,}")

print(f"\nColumns: {raw_dataset.column_names}")
print(f"\nExample:")
for k, v in raw_dataset[0].items():
    print(f"  {k}: {v[:200] if isinstance(v, str) else v}")

In [ ]:
# ---------------------------------------------------------------------------
# Format with Alpaca template + train/eval split
# ---------------------------------------------------------------------------
ALPACA_TEMPLATE = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

eos_token = tokenizer.eos_token


def format_alpaca(example):
    text = ALPACA_TEMPLATE.format(
        instruction=example["instruction"],
        input=example.get("input", ""),
        output=example["output"],
    )
    return {"text": text + eos_token}


formatted_dataset = raw_dataset.map(format_alpaca)

# Train / eval split
split = formatted_dataset.train_test_split(test_size=test_size, seed=seed)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train samples: {len(train_dataset):,}")
print(f"Eval samples:  {len(eval_dataset):,}")
print(f"\nFormatted example (first 500 chars):")
print(train_dataset[0]["text"][:500])

## 4 — Token Length Analysis

Visualize the distribution of tokenized sequence lengths to verify
that `max_seq_length` is appropriate for this dataset.

In [ ]:
# ---------------------------------------------------------------------------
# Token length distribution
# ---------------------------------------------------------------------------
sample_texts = train_dataset["text"][:500]
token_lengths = [len(tokenizer.encode(t)) for t in sample_texts]

print(f"Token length stats (sample of {len(sample_texts)}):")
print(f"  Min:    {min(token_lengths)}")
print(f"  Max:    {max(token_lengths)}")
print(f"  Mean:   {np.mean(token_lengths):.0f}")
print(f"  Median: {np.median(token_lengths):.0f}")
print(f"  95th %%: {np.percentile(token_lengths, 95):.0f}")
print(f"  > max_seq_length ({max_seq_length}): {sum(1 for l in token_lengths if l > max_seq_length)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(token_lengths, bins=50, color="#2196F3", edgecolor="white", alpha=0.8)
axes[0].axvline(max_seq_length, color="red", linestyle="--", linewidth=2,
               label=f"max_seq_length={max_seq_length}")
axes[0].set_xlabel("Token Length")
axes[0].set_ylabel("Count")
axes[0].set_title("Token Length Distribution")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative distribution
sorted_lengths = np.sort(token_lengths)
cumulative = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths)
axes[1].plot(sorted_lengths, cumulative, color="#9C27B0", linewidth=2)
axes[1].axvline(max_seq_length, color="red", linestyle="--", linewidth=2,
               label=f"max_seq_length={max_seq_length}")
axes[1].set_xlabel("Token Length")
axes[1].set_ylabel("Cumulative Fraction")
axes[1].set_title("Cumulative Token Length Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(f"{plots_dir}/token_length_distribution.png", dpi=120)
plt.show()
print(f"Saved \u2192 {plots_dir}/token_length_distribution.png")

## 5 — Training

Train the LoRA adapters using the SFTTrainer from TRL. The base model
weights remain frozen — only the low-rank adapter matrices are updated.

In [ ]:
# ---------------------------------------------------------------------------
# Configure training
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="outputs/training_checkpoints",
    max_steps=max_steps,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    warmup_steps=warmup_steps,
    logging_steps=logging_steps,
    lr_scheduler_type=lr_scheduler_type,
    optim="adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=seed,
    eval_strategy="steps",
    eval_steps=logging_steps,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
)

print("Training configuration:")
print(f"  Max steps: {max_steps}")
print(f"  Batch size: {per_device_train_batch_size}")
print(f"  Gradient accumulation: {gradient_accumulation_steps}")
print(f"  Effective batch size: {per_device_train_batch_size * gradient_accumulation_steps}")
print(f"  Learning rate: {learning_rate}")
print(f"  Scheduler: {lr_scheduler_type}")
print(f"  Precision: {'bf16' if training_args.bf16 else 'fp16'}")

In [ ]:
# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
gpu_mem_before = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0

train_start = time.time()
train_result = trainer.train()
train_elapsed = time.time() - train_start

gpu_mem_after = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

print(f"\nTraining complete in {train_elapsed:.1f}s")
print(f"  Final train loss: {train_result.training_loss:.4f}")
print(f"  GPU memory before: {gpu_mem_before:.2f} GB")
print(f"  GPU peak memory:   {gpu_mem_after:.2f} GB")

## 6 — Training Loss Visualization

In [ ]:
# ---------------------------------------------------------------------------
# Plot training loss + learning rate schedule
# ---------------------------------------------------------------------------
log_history = trainer.state.log_history

train_steps = [e["step"] for e in log_history if "loss" in e]
train_losses = [e["loss"] for e in log_history if "loss" in e]
lr_steps = [e["step"] for e in log_history if "learning_rate" in e]
lr_values = [e["learning_rate"] for e in log_history if "learning_rate" in e]

fig, ax1 = plt.subplots(figsize=(10, 5))

# Loss
ax1.plot(train_steps, train_losses, color="#2196F3", alpha=0.4, linewidth=1, label="Train loss")
if len(train_losses) >= 3:
    window = min(5, len(train_losses))
    rolling = np.convolve(train_losses, np.ones(window) / window, mode="valid")
    rolling_steps = train_steps[window - 1:]
    ax1.plot(rolling_steps, rolling, color="#4CAF50", linewidth=2, label=f"Rolling mean (w={window})")
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper left")

# Learning rate on secondary axis
if lr_values:
    ax2 = ax1.twinx()
    ax2.plot(lr_steps, lr_values, color="#FF9800", linewidth=1.5, linestyle="--", label="Learning rate")
    ax2.set_ylabel("Learning Rate")
    ax2.legend(loc="upper right")

ax1.set_title("Training Loss & Learning Rate")
fig.tight_layout()
fig.savefig(f"{plots_dir}/training_loss.png", dpi=120)
plt.show()
print(f"Saved \u2192 {plots_dir}/training_loss.png")

## 7 — Evaluation

Evaluate the fine-tuned model on the held-out set and compute perplexity.

In [ ]:
# ---------------------------------------------------------------------------
# Evaluate on held-out set
# ---------------------------------------------------------------------------
eval_result = trainer.evaluate()
eval_loss = eval_result["eval_loss"]
perplexity = math.exp(eval_loss)

# Get final train loss from log history
final_train_loss = train_result.training_loss

print(f"Eval loss:    {eval_loss:.4f}")
print(f"Perplexity:   {perplexity:.2f}")
print(f"Train loss:   {final_train_loss:.4f}")

# Bar chart: train vs eval loss
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Train Loss", "Eval Loss"],
    [final_train_loss, eval_loss],
    color=["#2196F3", "#FF9800"],
    edgecolor="white",
)
for bar, val in zip(bars, [final_train_loss, eval_loss]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", fontsize=11)
ax.set_ylabel("Loss")
ax.set_title("Train vs Eval Loss")
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(f"{plots_dir}/train_vs_eval_loss.png", dpi=120)
plt.show()
print(f"Saved \u2192 {plots_dir}/train_vs_eval_loss.png")

## 8 — Sample Generation

Generate responses from the fine-tuned model to qualitatively assess
instruction-following ability.

In [ ]:
# ---------------------------------------------------------------------------
# Generate sample responses
# ---------------------------------------------------------------------------
FastLanguageModel.for_inference(model)

generated_samples = []

for i, prompt_text in enumerate(test_prompts):
    formatted_prompt = ALPACA_TEMPLATE.format(
        instruction=prompt_text,
        input="",
        output="",
    )
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    generated_samples.append({"prompt": prompt_text, "response": response.strip()})

    print(f"\n{'=' * 60}")
    print(f"Prompt {i + 1}: {prompt_text}")
    print(f"{'─' * 60}")
    print(response.strip())

# Response length chart
response_lengths = [len(s["response"]) for s in generated_samples]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(response_lengths)), response_lengths, color="#4CAF50", edgecolor="white")
ax.set_xlabel("Prompt Index")
ax.set_ylabel("Response Length (chars)")
ax.set_title("Response Lengths")
ax.set_xticks(range(len(response_lengths)))
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(f"{plots_dir}/response_lengths.png", dpi=120)
plt.show()
print(f"\nSaved \u2192 {plots_dir}/response_lengths.png")

## 9 — Save Model

Save the LoRA adapters (small, portable) and optionally merge them
into a full 16-bit model for direct inference without PEFT.

In [ ]:
# ---------------------------------------------------------------------------
# Save LoRA adapters
# ---------------------------------------------------------------------------
Path(model_output_path).mkdir(parents=True, exist_ok=True)
model.save_pretrained(model_output_path)
tokenizer.save_pretrained(model_output_path)

adapter_size = sum(
    f.stat().st_size for f in Path(model_output_path).rglob("*") if f.is_file()
) / 1e6
print(f"LoRA adapters saved \u2192 {model_output_path} ({adapter_size:.1f} MB)")

In [ ]:
# ---------------------------------------------------------------------------
# Optionally save merged 16-bit model
# ---------------------------------------------------------------------------
merged_size_mb = None
if save_merged_model:
    Path(merged_output_path).mkdir(parents=True, exist_ok=True)
    model.save_pretrained_merged(
        merged_output_path,
        tokenizer,
        save_method="merged_16bit",
    )
    merged_size_mb = sum(
        f.stat().st_size for f in Path(merged_output_path).rglob("*") if f.is_file()
    ) / 1e6
    print(f"Merged model saved \u2192 {merged_output_path} ({merged_size_mb:.1f} MB)")
else:
    print("Skipping merged model save (save_merged_model=False)")

# Next step: for GGUF export, use:
# model.save_pretrained_gguf("outputs/models/gguf", tokenizer, quantization_method="q4_k_m")

## 10 — Summary

In [ ]:
# ---------------------------------------------------------------------------
# Write metrics JSON
# ---------------------------------------------------------------------------
gpu_info = {}
if torch.cuda.is_available():
    gpu_info = {
        "device_name": torch.cuda.get_device_name(0),
        "total_vram_gb": round(torch.cuda.get_device_properties(0).total_mem / 1e9, 2),
        "peak_memory_gb": round(gpu_mem_after, 2),
    }

metrics_output = {
    "run_metadata": {
        "timestamp": run_start.isoformat(),
        "model_name": model_name,
        "dataset_name": dataset_name,
        "dataset_sample_size": dataset_sample_size,
        "max_seq_length": max_seq_length,
        "load_in_4bit": load_in_4bit,
        "seed": seed,
        "n_train": len(train_dataset),
        "n_eval": len(eval_dataset),
    },
    "lora_config": {
        "r": lora_r,
        "alpha": lora_alpha,
        "dropout": lora_dropout,
        "target_modules": target_modules,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "trainable_pct": round(pct, 2),
    },
    "training": {
        "max_steps": max_steps,
        "per_device_train_batch_size": per_device_train_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": learning_rate,
        "lr_scheduler_type": lr_scheduler_type,
        "warmup_steps": warmup_steps,
        "train_loss": round(final_train_loss, 4),
        "train_runtime_s": round(train_elapsed, 1),
    },
    "evaluation": {
        "eval_loss": round(eval_loss, 4),
        "perplexity": round(perplexity, 2),
    },
    "model_outputs": {
        "lora_adapter_path": model_output_path,
        "lora_adapter_size_mb": round(adapter_size, 1),
        "merged_model_path": merged_output_path if save_merged_model else None,
        "merged_model_size_mb": round(merged_size_mb, 1) if merged_size_mb else None,
    },
    "generated_samples": generated_samples,
    "gpu_info": gpu_info,
}

Path(metrics_json_path).parent.mkdir(parents=True, exist_ok=True)
with open(metrics_json_path, "w") as f:
    json.dump(metrics_output, f, indent=2, default=str)

print(f"Metrics JSON saved \u2192 {metrics_json_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------
print("=" * 60)
print("RUN COMPLETE")
print("=" * 60)
print(f"  Model:              {model_name}")
print(f"  Dataset:            {dataset_name} ({len(train_dataset) + len(eval_dataset)} samples)")
print(f"  LoRA rank:          {lora_r}")
print(f"  Trainable params:   {trainable_params:,} ({pct:.2f}%)")
print(f"  Training steps:     {max_steps}")
print(f"  Train loss:         {final_train_loss:.4f}")
print(f"  Eval loss:          {eval_loss:.4f}")
print(f"  Perplexity:         {perplexity:.2f}")
print(f"  Training time:      {train_elapsed:.1f}s")
print(f"  LoRA adapters:      {model_output_path}")
if save_merged_model:
    print(f"  Merged model:       {merged_output_path}")
print(f"  Metrics JSON:       {metrics_json_path}")
print(f"  Plots:              {plots_dir}")
if executed_notebook_path:
    print(f"  Executed notebook:  {executed_notebook_path}")
print("=" * 60)